# 07 · Production patterns — make TravelMind survive real traffic
### orchestrator pool · retries & backoff · bounded context · concurrency-safe sessions · observability · validation & guardrails · readiness · cost control

The capstone (notebook 06) works in a demo and cuts corners that bite under load. This notebook fixes each one, from scratch to a deployable `app.py`.

| Corner the capstone cut | What breaks under load | Fix here |
|---|---|---|
| Rebuilds the orchestrator every call | Wasted CPU, slower p99, no reuse | **Orchestrator pool** (bounded LRU + TTL, thread-safe) |
| No retry strategy | 424 / throttle errors when many sessions hit at once | **Adaptive SDK retries + app-level backoff** |
| Conversation grows unbounded | Context overflow, rising cost/latency | **Sliding-window conversation manager** |
| In-RAM session only | State lost on restart; races across instances | **S3 session manager** (file-based has no locking) |
| No visibility | You can't see latency, tokens, or which request failed | **Structured observability + correlation IDs** |
| Trusts the payload, crashes on tool failure | One bad request takes down the handler | **Input validation + graceful degradation** |
| Cold start serves the first user slowly | First request after deploy is sluggish | **Readiness gating + warmup + busy status** |
| One model for everything; no cost ceiling | Overspend on trivial requests | **Cost-aware routing + prompt caching + per-request usage** |
| No content safety | Unfiltered in/out | **Bedrock Guardrails** |

```mermaid
flowchart TB
    REQ[Request + context] --> V[Validate payload]
    V -->|bad| ERR[Structured error + correlation id]
    V -->|ok| TIER[Pick model tier]
    TIER --> POOL[Orchestrator pool<br/>key = actor + session + tier]
    POOL --> RETRY[call_with_retry<br/>throttle backoff]
    RETRY --> RESP[Result + usage + correlation id]
    POOL -. on miss .-> BUILD[Build: resilient model + window + S3 session + memory hook]
    PING[/ping: HEALTHY only after warmup/] -.-> POOL
```

> Prereqs: notebooks 00–06. Region `us-east-1`; Haiku 4.5 (cheap tier) and Sonnet 4.5 (escalation). Construction is verified against the installed packages; AWS calls need your creds, model access, and (optionally) an S3 bucket + a Guardrail.


## Config — everything from the environment

Nothing hardcoded. Secrets live in Identity (notebook 04); ids and tunables come from env vars so the same image runs in dev and prod.


In [ ]:
import os, time, random, logging, threading
from collections import OrderedDict

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
REGION        = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
MEMORY_ID     = os.environ.get("MEMORY_ID")             # from notebook 03/06 (optional -> degrades)
SESSION_BUCKET= os.environ.get("SESSION_BUCKET")        # S3 bucket for session persistence (optional)
GUARDRAIL_ID  = os.environ.get("GUARDRAIL_ID")          # Bedrock Guardrail (optional)
GUARDRAIL_VER = os.environ.get("GUARDRAIL_VERSION", "DRAFT")

MODEL_CHEAP = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_STRONG= "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Pool tunables
POOL_MAX_SIZE = int(os.environ.get("POOL_MAX_SIZE", "200"))   # max live conversations cached
POOL_TTL_SEC  = int(os.environ.get("POOL_TTL_SEC", "900"))    # evict idle conversations after 15 min

# Per-request budget + pricing (SET to current Bedrock pricing; left at 0 so nothing is faked)
MAX_TOKENS_PER_REQ = int(os.environ.get("MAX_TOKENS_PER_REQ", "1024"))
PRICE_PER_1K_IN  = float(os.environ.get("PRICE_PER_1K_IN", "0.0"))   # fill from current Bedrock pricing
PRICE_PER_1K_OUT = float(os.environ.get("PRICE_PER_1K_OUT", "0.0"))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("travelmind")
print("config loaded; region:", REGION, "| memory:", bool(MEMORY_ID), "| s3 sessions:", bool(SESSION_BUCKET), "| guardrail:", bool(GUARDRAIL_ID))

## Step 1 — a resilient model factory

One place that builds every model with the production knobs:

- **Adaptive retries + timeouts** via a botocore `Config` injected through `boto_client_config`. Adaptive mode backs off automatically when Bedrock throttles — your first line of defense against the 424s under cohort load.
- **Prompt caching** (`cache_prompt`, `cache_tools`): a large, stable system prompt and tool spec get cached on Bedrock, so repeated calls reuse them — lower cost and latency.
- **Guardrails** (`guardrail_id`/`guardrail_version`): input/output content safety, applied at the model layer.
- A **token ceiling** (`max_tokens`) so no single call runs away.


In [ ]:
from strands.models import BedrockModel
from botocore.config import Config

def make_model(model_id: str, cache: bool = True) -> BedrockModel:
    cfg = Config(
        retries={"max_attempts": 5, "mode": "adaptive"},   # SDK-layer backoff on throttling
        read_timeout=120, connect_timeout=10,
    )
    kwargs = dict(model_id=model_id, region_name=REGION, boto_client_config=cfg, max_tokens=MAX_TOKENS_PER_REQ)
    if cache:
        kwargs.update(cache_prompt="default", cache_tools="default")   # reuse stable prompt/tools across calls
    if GUARDRAIL_ID:
        kwargs.update(guardrail_id=GUARDRAIL_ID, guardrail_version=GUARDRAIL_VER)
    return BedrockModel(**kwargs)

print("model factory ready (retries + caching + optional guardrail)")

## Step 2 — bounded context + concurrency-safe sessions

Two different jobs people confuse:

- **Conversation manager** keeps the *in-prompt* history from growing forever. `SlidingWindowConversationManager(window_size=N)` keeps the last N messages; `SummarizingConversationManager` summarizes older ones instead of dropping them. Without this you eventually hit a context-overflow error and pay for ever-larger prompts.
- **Session manager** *persists* the conversation so a restarted or different instance can resume it. `S3SessionManager` is safe for concurrent instances; **`FileSessionManager` has no locking** — fine for one local process, wrong for production. This is separate from AgentCore **Memory** (notebook 03), which stores *extracted* long-term knowledge (preferences/facts), not the raw transcript.


In [ ]:
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.session import S3SessionManager

def make_conversation_manager():
    return SlidingWindowConversationManager(window_size=40, should_truncate_results=True)

def make_session_manager(actor_id: str, session_id: str):
    if not SESSION_BUCKET:
        return None                                   # degrade: in-RAM only (dev)
    return S3SessionManager(
        session_id=f"{actor_id}:{session_id}",        # one persisted transcript per conversation
        bucket=SESSION_BUCKET, prefix="travelmind/", region_name=REGION,
    )

print("context-window + session-manager factories ready")

## Step 3 — the orchestrator pool (the headline)

Building the orchestrator graph on every call is waste. Cache it. The non-obvious part is **the key**.

- Key by **`(actor_id, session_id, tier)`** — i.e. one orchestrator per *conversation* (and model tier).
- **Why not key by actor only?** The Strands `Agent` accumulates in-memory message history. Pooling purely by actor would let two of that user's separate conversations share one agent and bleed context across them. Conversation is the correct isolation unit.
- **Bounded LRU** so memory can't grow without limit; **TTL** so an idle conversation's cached orchestrator (and its preloaded preferences) eventually refresh.
- **Thread-safe**: Runtime can reuse one warm process across requests. We hold the lock only during *construction* (fast); the actual model call happens outside the lock, so different conversations run in parallel.

The factory is injected, which also makes the pool testable without touching AWS.


In [ ]:
class OrchestratorPool:
    """Thread-safe, bounded, TTL-evicting cache of orchestrators keyed by (actor, session, tier)."""
    def __init__(self, factory, max_size: int, ttl_sec: int):
        self._factory = factory
        self._max, self._ttl = max_size, ttl_sec
        self._store = OrderedDict()                 # key -> (orchestrator, last_used_ts)
        self._lock = threading.RLock()
        self.builds = 0                             # observability: how many cold builds happened

    def _purge_expired(self, now: float):
        stale = [k for k, (_, ts) in self._store.items() if now - ts > self._ttl]
        for k in stale:
            self._store.pop(k, None)

    def get_or_create(self, actor_id: str, session_id: str, tier: str = "cheap"):
        key = (actor_id, session_id, tier)
        now = time.time()
        with self._lock:                            # held only for construction, not the model call
            self._purge_expired(now)
            if key in self._store:
                orch, _ = self._store.pop(key)
                self._store[key] = (orch, now)      # mark most-recently-used
                return orch
            orch = self._factory(actor_id, session_id, tier)   # cold build
            self.builds += 1
            self._store[key] = (orch, now)
            while len(self._store) > self._max:     # evict least-recently-used
                self._store.popitem(last=False)
            return orch

    def size(self) -> int:
        with self._lock:
            return len(self._store)

The pool's factory wires a full orchestrator: resilient model (by tier), bounded context, S3 session, and the memory hook — each degrading gracefully if its backing service isn't configured.


In [ ]:
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, AgentInitializedEvent

# --- specialists built once (shared, stateless tools) ---
@tool
def flight_status(flight_number: str) -> dict:
    """Current status of a flight by its number (e.g. BA117)."""
    return {"BA117": {"status": "Cancelled", "from": "LHR", "to": "JFK", "eu261_band_eur": 600}}\
        .get(flight_number.upper(), {"status": "Unknown"})

@tool
def find_alt_flight(origin: str, dest: str) -> list:
    """Find alternative flights between two airport codes."""
    return [{"flight": "BA179", "from": origin, "to": dest, "price_gbp": 420, "departs_in_h": 4}]

@tool
def find_hotel(city: str, nights: int) -> list:
    """Find hotels in a city for a number of nights."""
    return [{"name": "JFK Airport Inn", "city": city, "per_night_gbp": 150, "nights": nights}]

flight_agent = Agent(model=make_model(MODEL_CHEAP), name="flight_agent",
                     tools=[flight_status, find_alt_flight],
                     system_prompt="You handle flights only: status and alternatives. Be concise.")
hotel_agent  = Agent(model=make_model(MODEL_CHEAP), name="hotel_agent", tools=[find_hotel],
                     system_prompt="You handle hotels only. Be concise.")
policy_agent = Agent(model=make_model(MODEL_CHEAP), name="policy_agent",
                     system_prompt="Explain EU261 compensation rules plainly and briefly.")

@tool
def ask_flight_team(query: str) -> str:
    """Delegate a flight question to the flight specialist (degrades to a clear message on failure)."""
    try:
        return str(flight_agent(query))
    except Exception as e:
        log.warning("flight specialist failed: %s", e)
        return "Flight service is temporarily unavailable."

@tool
def ask_hotel_team(query: str) -> str:
    """Delegate a hotel question to the hotel specialist (degrades on failure)."""
    try:
        return str(hotel_agent(query))
    except Exception as e:
        log.warning("hotel specialist failed: %s", e)
        return "Hotel service is temporarily unavailable."

@tool
def ask_policy_team(query: str) -> str:
    """Delegate a compensation/policy question to the policy specialist (degrades on failure)."""
    try:
        return str(policy_agent(query))
    except Exception as e:
        log.warning("policy specialist failed: %s", e)
        return "Policy service is temporarily unavailable."

# --- memory hook (from notebook 03) ---
def _text_of(message) -> str:
    return "".join(b.get("text", "") for b in message.get("content", []) if isinstance(b, dict)).strip()

class MemoryHook(HookProvider):
    def __init__(self, client, memory_id, actor_id, session_id):
        self.client, self.memory_id = client, memory_id
        self.actor_id, self.session_id = actor_id, session_id
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AgentInitializedEvent, self.on_init)
        registry.add_callback(MessageAddedEvent, self.on_message)
    def on_init(self, event):
        try:
            recalled = self.client.retrieve_memories(
                memory_id=self.memory_id, namespace=f"travelmind/{self.actor_id}/preferences",
                query="seat and meal preferences", top_k=5)
            if recalled:
                event.agent.system_prompt += f"\n\nKnown traveler preferences: {recalled}"
        except Exception as e:
            log.warning("memory preload skipped: %s", e)        # degrade, don't crash
    def on_message(self, event):
        role = event.message.get("role", "").upper(); text = _text_of(event.message)
        if role in ("USER", "ASSISTANT") and text:
            try:
                self.client.create_event(memory_id=self.memory_id, actor_id=self.actor_id,
                                         session_id=self.session_id, messages=[(text, role)])
            except Exception as e:
                log.warning("memory write skipped: %s", e)

# optional memory client
_mc = None
if MEMORY_ID:
    from bedrock_agentcore.memory import MemoryClient
    _mc = MemoryClient(region_name=REGION)

def build_orchestrator(actor_id: str, session_id: str, tier: str):
    model = make_model(MODEL_STRONG if tier == "strong" else MODEL_CHEAP)
    hooks = [MemoryHook(_mc, MEMORY_ID, actor_id, session_id)] if _mc else []
    return Agent(
        model=model,
        tools=[ask_flight_team, ask_policy_team, ask_hotel_team],
        system_prompt=("You are TravelMind. For a disruption: confirm status, state exact compensation, "
                       "offer an alternative + hotel, honor known preferences. Delegate, then answer once."),
        conversation_manager=make_conversation_manager(),
        session_manager=make_session_manager(actor_id, session_id),
        hooks=hooks,
    )

POOL = OrchestratorPool(build_orchestrator, max_size=POOL_MAX_SIZE, ttl_sec=POOL_TTL_SEC)
print("pool + factory wired; specialists ready")

## Step 4 — resilient invocation (two layers of retry)

The SDK already retries on throttling (adaptive mode, Step 1). On top of that, an **app-level** wrapper gives you a hard attempt ceiling, jittered exponential backoff, and a place to log/measure. Jitter matters: without it, a cohort that all throttle at once retries in lockstep and re-throttles together.

We catch `ModelThrottledException` to back off, and treat `ContextWindowOverflowException` as a degrade signal (the window manager should prevent it).


In [ ]:
from strands.types.exceptions import ModelThrottledException, ContextWindowOverflowException

def call_with_retry(agent, prompt: str, max_attempts: int = 4, base: float = 0.5, cap: float = 8.0):
    """Invoke an agent with jittered exponential backoff on throttling."""
    for attempt in range(max_attempts):
        try:
            return agent(prompt)
        except ModelThrottledException:
            if attempt == max_attempts - 1:
                raise
            sleep = min(cap, base * (2 ** attempt)) + random.uniform(0, 0.25)   # jitter avoids lockstep retries
            log.warning("throttled; backing off %.2fs (attempt %d)", sleep, attempt + 1)
            time.sleep(sleep)
        except ContextWindowOverflowException:
            log.error("context overflow despite window manager — degrading")
            raise

## Step 5 — input validation (and where guardrails fit)

Never trust the payload. Validate shape, types, and size *before* spending a model call. Reject clearly. Content safety is layered: **structural** validation here, **semantic** safety via the Guardrail wired into `make_model` (Step 1).


In [ ]:
MAX_PROMPT_CHARS = 4000

def validate_payload(payload: dict):
    """Return (prompt, actor_id, session_id) or raise ValueError with a clear reason."""
    if not isinstance(payload, dict):
        raise ValueError("payload must be a JSON object")
    prompt = payload.get("prompt")
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("'prompt' is required and must be a non-empty string")
    if len(prompt) > MAX_PROMPT_CHARS:
        raise ValueError(f"'prompt' exceeds {MAX_PROMPT_CHARS} characters")
    actor   = str(payload.get("actor_id", "anonymous"))[:128]
    session = str(payload.get("session_id", f"session-{actor}"))[:128]
    return prompt.strip(), actor, session

## Step 6 — readiness, warmup, and busy status

Two platform signals via `/ping`:

- **Readiness:** report `HEALTHY` only after warmup finishes, so traffic doesn't hit a cold process. Use `@app.ping` to return your own status.
- **Busy:** while a long background job runs, report `HEALTHY_BUSY` so the platform knows the instance is alive but loaded. `add_async_task` / `complete_async_task` do this automatically; `force_ping_status` does it manually.

Warmup pre-builds one orchestrator (and in real code, fires a tiny model ping) so the first user request isn't the one paying the JIT cost.


In [ ]:
from bedrock_agentcore import BedrockAgentCoreApp, PingStatus

# (illustrative — the deployable version lives in app.py below)
_demo_app = BedrockAgentCoreApp()
_READY = {"ok": False}

@_demo_app.ping
def _ping():
    return PingStatus.HEALTHY if _READY["ok"] else PingStatus.HEALTHY_BUSY

def warmup():
    try:
        POOL.get_or_create("warmup", "warmup", "cheap")   # construct once so the path is hot
    except Exception as e:
        log.warning("warmup build issue: %s", e)
    finally:
        _READY["ok"] = True

print("readiness gate + warmup defined")

## Step 7 — cost-aware routing + per-request usage

Don't pay Sonnet prices for "is BA117 on time?". Route cheap by default; escalate only when the request looks complex (multiple intents). After the call, read the real token usage from `result.metrics.accumulated_usage` (it even exposes prompt-cache hits) to log cost and catch budget overruns.

A heuristic router is free; a tiny Haiku classifier is more accurate but costs a call. Start with the heuristic.


In [ ]:
def pick_model(prompt: str) -> str:
    """Cheap heuristic: escalate to the strong model only for multi-intent / complex asks."""
    p = prompt.lower()
    signals = ["and", "also", "compensation", "rebook", "alternative", "hotel", "plus", "then"]
    hits = sum(s in p for s in signals)
    return "strong" if (hits >= 2 or len(prompt) > 240) else "cheap"

def usage_summary(result) -> dict:
    m = getattr(result, "metrics", None)
    u = dict(getattr(m, "accumulated_usage", {}) or {}) if m else {}
    cost = (u.get("inputTokens", 0) / 1000.0) * PRICE_PER_1K_IN + \
           (u.get("outputTokens", 0) / 1000.0) * PRICE_PER_1K_OUT
    return {"input": u.get("inputTokens"), "output": u.get("outputTokens"),
            "total": u.get("totalTokens"), "cache_read": u.get("cacheReadInputTokens"),
            "est_cost": round(cost, 6)}      # est_cost is 0 until you set current Bedrock pricing

## Step 8 — the hardened `app.py`

Everything assembled. The entrypoint takes `(payload, context)` — `context.session_id` is a ready-made correlation id. Flow: validate → route → pool → retry → structured response with usage. Failures degrade to a clean error, never an unhandled crash.


In [ ]:
%%writefile app.py
"""TravelMind — production-hardened AgentCore Runtime app."""
import os, time, random, logging, threading
from collections import OrderedDict
from botocore.config import Config

from bedrock_agentcore import BedrockAgentCoreApp, PingStatus
from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.session import S3SessionManager
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, AgentInitializedEvent
from strands.types.exceptions import ModelThrottledException, ContextWindowOverflowException

REGION         = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
MEMORY_ID      = os.environ.get("MEMORY_ID")
SESSION_BUCKET = os.environ.get("SESSION_BUCKET")
GUARDRAIL_ID   = os.environ.get("GUARDRAIL_ID")
GUARDRAIL_VER  = os.environ.get("GUARDRAIL_VERSION", "DRAFT")
MODEL_CHEAP    = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_STRONG   = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
POOL_MAX_SIZE  = int(os.environ.get("POOL_MAX_SIZE", "200"))
POOL_TTL_SEC   = int(os.environ.get("POOL_TTL_SEC", "900"))
MAX_TOKENS     = int(os.environ.get("MAX_TOKENS_PER_REQ", "1024"))
MAX_PROMPT     = 4000

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("travelmind")

def make_model(model_id, cache=True):
    cfg = Config(retries={"max_attempts": 5, "mode": "adaptive"}, read_timeout=120, connect_timeout=10)
    kw = dict(model_id=model_id, region_name=REGION, boto_client_config=cfg, max_tokens=MAX_TOKENS)
    if cache:        kw.update(cache_prompt="default", cache_tools="default")
    if GUARDRAIL_ID: kw.update(guardrail_id=GUARDRAIL_ID, guardrail_version=GUARDRAIL_VER)
    return BedrockModel(**kw)

@tool
def flight_status(flight_number: str) -> dict:
    """Current status of a flight by its number (e.g. BA117)."""
    return {"BA117": {"status": "Cancelled", "from": "LHR", "to": "JFK", "eu261_band_eur": 600}}\
        .get(flight_number.upper(), {"status": "Unknown"})
@tool
def find_alt_flight(origin: str, dest: str) -> list:
    """Find alternative flights between two airport codes."""
    return [{"flight": "BA179", "from": origin, "to": dest, "price_gbp": 420, "departs_in_h": 4}]
@tool
def find_hotel(city: str, nights: int) -> list:
    """Find hotels in a city for a number of nights."""
    return [{"name": "JFK Airport Inn", "city": city, "per_night_gbp": 150, "nights": nights}]

_flight = Agent(model=make_model(MODEL_CHEAP), name="flight_agent", tools=[flight_status, find_alt_flight],
                system_prompt="Flights only: status and alternatives. Concise.")
_hotel  = Agent(model=make_model(MODEL_CHEAP), name="hotel_agent", tools=[find_hotel],
                system_prompt="Hotels only. Concise.")
_policy = Agent(model=make_model(MODEL_CHEAP), name="policy_agent",
                system_prompt="Explain EU261 compensation plainly and briefly.")

def _safe(agent, q, label):
    try: return str(agent(q))
    except Exception as e:
        log.warning("%s specialist failed: %s", label, e); return f"{label} service temporarily unavailable."
@tool
def ask_flight_team(query: str) -> str:
    """Delegate a flight question to the flight specialist."""
    return _safe(_flight, query, "flight")
@tool
def ask_hotel_team(query: str) -> str:
    """Delegate a hotel question to the hotel specialist."""
    return _safe(_hotel, query, "hotel")
@tool
def ask_policy_team(query: str) -> str:
    """Delegate a compensation/policy question to the policy specialist."""
    return _safe(_policy, query, "policy")

def _text_of(m): return "".join(b.get("text","") for b in m.get("content",[]) if isinstance(b, dict)).strip()
class MemoryHook(HookProvider):
    def __init__(self, c, m, a, s): self.client=c; self.memory_id=m; self.actor_id=a; self.session_id=s
    def register_hooks(self, r):
        r.add_callback(AgentInitializedEvent, self.on_init); r.add_callback(MessageAddedEvent, self.on_message)
    def on_init(self, e):
        try:
            rec = self.client.retrieve_memories(memory_id=self.memory_id,
                  namespace=f"travelmind/{self.actor_id}/preferences", query="seat and meal preferences", top_k=5)
            if rec: e.agent.system_prompt += f"\n\nKnown traveler preferences: {rec}"
        except Exception as ex: log.warning("memory preload skipped: %s", ex)
    def on_message(self, e):
        role = e.message.get("role","").upper(); text = _text_of(e.message)
        if role in ("USER","ASSISTANT") and text:
            try: self.client.create_event(memory_id=self.memory_id, actor_id=self.actor_id,
                                           session_id=self.session_id, messages=[(text, role)])
            except Exception as ex: log.warning("memory write skipped: %s", ex)

_mc = None
if MEMORY_ID:
    from bedrock_agentcore.memory import MemoryClient
    _mc = MemoryClient(region_name=REGION)

def _session_mgr(actor, session):
    if not SESSION_BUCKET: return None
    return S3SessionManager(session_id=f"{actor}:{session}", bucket=SESSION_BUCKET, prefix="travelmind/", region_name=REGION)

def build_orchestrator(actor, session, tier):
    hooks = [MemoryHook(_mc, MEMORY_ID, actor, session)] if _mc else []
    return Agent(model=make_model(MODEL_STRONG if tier=="strong" else MODEL_CHEAP),
                 tools=[ask_flight_team, ask_policy_team, ask_hotel_team],
                 system_prompt=("You are TravelMind. For a disruption: confirm status, state exact compensation, "
                                "offer an alternative + hotel, honor known preferences. Delegate, then answer once."),
                 conversation_manager=SlidingWindowConversationManager(window_size=40),
                 session_manager=_session_mgr(actor, session), hooks=hooks)

class OrchestratorPool:
    def __init__(self, factory, max_size, ttl):
        self._f=factory; self._max=max_size; self._ttl=ttl
        self._store=OrderedDict(); self._lock=threading.RLock(); self.builds=0
    def get_or_create(self, actor, session, tier="cheap"):
        key=(actor, session, tier); now=time.time()
        with self._lock:
            for k in [k for k,(_,ts) in self._store.items() if now-ts>self._ttl]: self._store.pop(k, None)
            if key in self._store:
                o,_=self._store.pop(key); self._store[key]=(o, now); return o
            o=self._f(actor, session, tier); self.builds+=1; self._store[key]=(o, now)
            while len(self._store)>self._max: self._store.popitem(last=False)
            return o

POOL = OrchestratorPool(build_orchestrator, POOL_MAX_SIZE, POOL_TTL_SEC)

def pick_model(prompt):
    p=prompt.lower(); hits=sum(s in p for s in ["and","also","compensation","rebook","alternative","hotel","plus","then"])
    return "strong" if (hits>=2 or len(prompt)>240) else "cheap"

def call_with_retry(agent, prompt, max_attempts=4, base=0.5, cap=8.0):
    for attempt in range(max_attempts):
        try: return agent(prompt)
        except ModelThrottledException:
            if attempt==max_attempts-1: raise
            time.sleep(min(cap, base*(2**attempt)) + random.uniform(0, 0.25))
        except ContextWindowOverflowException:
            log.error("context overflow despite window manager"); raise

def validate(payload):
    if not isinstance(payload, dict): raise ValueError("payload must be a JSON object")
    prompt=payload.get("prompt")
    if not isinstance(prompt, str) or not prompt.strip(): raise ValueError("'prompt' required (non-empty string)")
    if len(prompt)>MAX_PROMPT: raise ValueError(f"'prompt' exceeds {MAX_PROMPT} chars")
    actor=str(payload.get("actor_id","anonymous"))[:128]; session=str(payload.get("session_id", f"session-{actor}"))[:128]
    return prompt.strip(), actor, session

app = BedrockAgentCoreApp()
_READY = {"ok": False}

@app.ping
def ping():
    return PingStatus.HEALTHY if _READY["ok"] else PingStatus.HEALTHY_BUSY

def _warmup():
    try: POOL.get_or_create("warmup","warmup","cheap")
    except Exception as e: log.warning("warmup issue: %s", e)
    finally: _READY["ok"] = True

@app.entrypoint
def invoke(payload, context):
    cid = getattr(context, "session_id", None) or "no-session"
    try:
        prompt, actor, session = validate(payload)
    except ValueError as e:
        return {"error": str(e), "correlation_id": cid}
    tier = pick_model(prompt)
    orch = POOL.get_or_create(actor, session, tier)
    t0 = time.time()
    try:
        result = call_with_retry(orch, prompt)
    except ModelThrottledException:
        return {"error": "service busy, please retry shortly", "correlation_id": cid}
    except Exception:
        log.exception("invocation failed cid=%s", cid)
        return {"error": "internal error", "correlation_id": cid}
    m = getattr(result, "metrics", None); usage = dict(getattr(m, "accumulated_usage", {}) or {}) if m else {}
    log.info("ok cid=%s tier=%s latency_ms=%d tokens=%s pool=%d builds=%d",
             cid, tier, int((time.time()-t0)*1000), usage.get("totalTokens"), len(POOL._store), POOL.builds)
    return {"result": str(result), "correlation_id": cid, "tier": tier, "usage": usage}

_warmup()

if __name__ == "__main__":
    app.run()

## Step 9 — prove the pool (offline, no AWS)

Hammer `get_or_create` from many threads with a **stub factory** that just counts builds. This verifies the three properties that matter — one build per key under concurrency, LRU eviction at the cap, TTL refresh — without spending a single model call.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

calls = {"n": 0}
def stub_factory(actor, session, tier):
    calls["n"] += 1
    time.sleep(0.01)                      # simulate construction cost
    return f"orch::{actor}:{session}:{tier}"

# A) one build per key under heavy concurrency
test_pool = OrchestratorPool(stub_factory, max_size=100, ttl_sec=60)
keys = [("amelia","s1","cheap"), ("bo","s1","cheap"), ("amelia","s2","cheap"), ("cara","s1","strong"), ("dan","s1","cheap")]
with ThreadPoolExecutor(max_workers=32) as ex:
    list(ex.map(lambda i: test_pool.get_or_create(*keys[i % len(keys)]), range(400)))
assert test_pool.builds == len(keys), f"expected {len(keys)} builds, got {test_pool.builds}"
print(f"A) concurrency: {test_pool.builds} builds for {len(keys)} unique conversations across 400 calls ✅")

# B) LRU eviction at the cap
small = OrchestratorPool(stub_factory, max_size=2, ttl_sec=60)
for i in range(5):
    small.get_or_create(f"user{i}", "s", "cheap")
assert small.size() == 2, small.size()
print(f"B) LRU: pool capped at {small.size()} (max_size=2) ✅")

# C) TTL refresh
ttl_pool = OrchestratorPool(stub_factory, max_size=10, ttl_sec=1)
ttl_pool.get_or_create("amelia", "s1", "cheap"); b1 = ttl_pool.builds
time.sleep(1.1)
ttl_pool.get_or_create("amelia", "s1", "cheap"); b2 = ttl_pool.builds
assert b2 == b1 + 1, (b1, b2)
print(f"C) TTL: idle conversation rebuilt after expiry (builds {b1} -> {b2}) ✅")
print("\nPOOL BEHAVIOUR VERIFIED ✅")

## Step 10 — deploy with production config

Thread the ids/tunables in as env vars. The execution role now needs more than Bedrock: **S3** (sessions), the **Guardrail**, and **Code Interpreter** if you re-add it. Least privilege still applies.


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

runtime = Runtime()
runtime.configure(
    entrypoint="app.py",
    agent_name="travelmind_prod",
    requirements=["strands-agents", "strands-agents-tools", "bedrock-agentcore"],
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region="us-east-1",
)
launch_result = runtime.launch(env_vars={
    "MEMORY_ID": MEMORY_ID or "",
    "SESSION_BUCKET": SESSION_BUCKET or "",
    "GUARDRAIL_ID": GUARDRAIL_ID or "",
    "POOL_MAX_SIZE": str(POOL_MAX_SIZE),
    "POOL_TTL_SEC": str(POOL_TTL_SEC),
})
print(launch_result)

In [ ]:
resp = runtime.invoke({
    "prompt": "BA117 LHR->JFK cancelled — compensation, an alternative, and a hotel near JFK for 1 night?",
    "actor_id": "traveler-amelia",
    "session_id": "trip-tokyo-001",
})
print(resp)   # note the correlation_id, tier, and usage in the response

## Step 11 — failure drills worth running

| Drill | How | Expected |
|---|---|---|
| Bad input | invoke `{"prompt": ""}` | clean `error` + correlation id, no crash, no model spend |
| Throttle | fire many concurrent invokes | adaptive + app retries absorb it; sustained load returns the "busy" message, not a 500 |
| Memory off | deploy without `MEMORY_ID` | agent still answers; logs show "memory preload skipped" |
| Specialist failure | make a tool raise | orchestrator returns a partial answer, not an exception |
| Cold start | invoke right after launch | `/ping` is `HEALTHY_BUSY` until warmup, then `HEALTHY`; first real call isn't cold |
| Pool reuse | invoke same `(actor, session)` repeatedly | logs show `builds` stays flat, `pool` size steady |


## Step 12 — what this still doesn't cover (be honest)

This is hardened, not bulletproof. Still out of scope:

- **Edge rate limiting / quotas per tenant** — do it at the gateway/API layer, before the agent.
- **Multi-region failover + blue/green deploys** — infra concern, via the CLI/IaC.
- **Secret rotation, PII redaction, audit trails** — Identity + Guardrails + your logging policy.
- **Eval gates in CI** — `agentcore add evaluator` / online-eval, fail the deploy on quality regressions.
- **Backpressure** — if the pool and downstream are saturated, shed load deliberately rather than queueing forever.

## Skeptic's corner

Every pattern here adds complexity and a new failure mode:

- The **pool** can serve a stale cached orchestrator with old preferences — that's why TTL exists; tune it to how often preferences change. And a conversation cached during a model call must not be invoked twice concurrently (a conversation is sequential by nature — don't fan out one conversation across threads).
- **Prompt caching** only pays off when the system prompt/tools are large and stable; for tiny prompts it's noise.
- **Retries** can amplify an outage if everyone retries — jitter and the attempt ceiling are not optional.
- **Cost routing** by heuristic will misroute sometimes; measure the misroute rate before trusting it, and consider a cheap classifier if it matters.

Add each one because a metric told you to, not because this notebook listed it. You now have the patterns *and* the judgement to apply them.
